# Pooling
:label:`sec_pooling`

- In many vision tasks, we care about **global information**:
  - E.g., *Does the image contain a cat?*
  - Thus, the **final layers** should be sensitive to the **entire input**.

- To achieve this, we:
  - **Gradually aggregate information** across layers,
  - Producing **coarser feature maps** as we go deeper,
  - While preserving the **benefits of convolutional layers** at intermediate steps.

- As we go deeper into the network:
  - The **receptive field** of each hidden unit becomes **larger**,
  - Especially when **spatial resolution is reduced**,
  - Since kernels cover a **larger effective area**.

- **Motivation for translation invariance**:
  - When detecting low-level features like **edges** (:numref:`sec_conv_layer`),
    - We want the representation to be **robust to small shifts** in the input.

- Example:
  - If `X` is an image with a sharp black-white boundary,
  - And we shift it by one pixel to get `Z[i, j] = X[i, j + 1]`,
  - The **edge position shifts**, leading to **different output**.
  - But in real settings, **objects rarely appear in the exact same location**:
    - Even with a tripod, **camera vibration** may shift pixels slightly.

- To handle this, we introduce **pooling layers**, which:
  - Help mitigate **sensitivity to small shifts**,
  - Perform **spatial downsampling** of the feature maps.


In [1]:
import torch
from torch import nn
from d2l import torch as d2l

## Maximum Pooling and Average Pooling

- Like convolutional layers, **pooling operators** use a **fixed-shape window**:
  - This window **slides over the input** using a defined **stride**.
  - At each location, the window computes **one output value**.

- The window used in pooling is often referred to as the **pooling window**.

- Key difference from convolutional layers:
  - Pooling layers have **no learnable parameters**.
  - There is **no kernel** involved.
  - The operations are **deterministic**.

- Common types of pooling:
  - **Maximum pooling (max-pooling)**:
    - Returns the **maximum value** in the pooling window.
  - **Average pooling**:
    - Returns the **average value** of elements in the pooling window.


- **Average pooling** dates back to the earliest CNNs:
  - Conceptually similar to **downsampling** an image.
  - Instead of picking every second or third pixel, we:
    - **Average adjacent pixels** to form a **lower-resolution image**.
    - This improves the **signal-to-noise ratio** by aggregating nearby information.

- **Max-pooling** was introduced by :citet:`Riesenhuber.Poggio.1999`:
  - In the context of **cognitive neuroscience**.
  - Describes how the brain might **hierarchically aggregate visual information** for object recognition.
  - An earlier version appeared in **speech recognition** :cite:`Yamaguchi.Sakamoto.Akabane.ea.1990`.

- In most practical applications, **max-pooling** is preferred over average pooling.

- Pooling operations follow the same **sliding window** structure as cross-correlation:
  - The **pooling window** starts at the **top-left** of the input tensor.
  - It **slides** from left to right, top to bottom.
  - At each position, it computes either:
    - The **maximum** (for max-pooling), or
    - The **average** (for average pooling)
    - Over the **subtensor** within the pooling window.

![Max-pooling with a pooling window shape of $2\times 2$. The shaded portions are the first output element as well as the input tensor elements used for the output computation: $\max(0, 1, 3, 4)=4$.](../img/pooling.svg)
:label:`fig_pooling`

- In :numref:`fig_pooling`, the **output tensor** has:
  - **Height = 2**, **Width = 2**
  - Each element is computed as the **maximum** in a $2 \times 2$ window:

  $$
  \begin{aligned}
  \max(0, 1, 3, 4) &= 4,\\
  \max(1, 2, 4, 5) &= 5,\\
  \max(3, 4, 6, 7) &= 7,\\
  \max(4, 5, 7, 8) &= 8.
  \end{aligned}
  $$


- More generally, we can define a **$p \times q$ pooling layer**:
  - It aggregates values over a region of size $p \times q$.

- Revisit the **edge detection** example:
  - Apply **$2 \times 2$ max-pooling** to the output of a convolutional layer.

- Let:
  - `X` be the **input** to the convolutional layer.
  - `Y` be the **output** of the pooling layer.

- Observation:
  - No matter the individual values of `X[i, j]`, `X[i, j + 1]`, `X[i+1, j]`, and `X[i+1, j + 1]`,
  - As long as at least one of them is 1, **max-pooling outputs** `Y[i, j] = 1`.

- Interpretation:
  - A **$2 \times 2$ max-pooling layer** can still detect a pattern,
    - Even if it **shifts by at most one element** in either height or width.
  - This introduces **local shift-invariance**.

- In the code below, we **implement forward propagation** of a pooling layer in `pool2d`:
  - The function resembles `corr2d` from :numref:`sec_conv_layer`,
  - But:
    - It **requires no kernel**,
    - It computes either:
      - The **maximum**, or
      - The **average** over each input region.


In [2]:
def pool2d(X, pool_size, mode='max'):
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] - p_h + 1, X.shape[1] - p_w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j] = X[i: i + p_h, j: j + p_w].max()
            elif mode == 'avg':
                Y[i, j] = X[i: i + p_h, j: j + p_w].mean()
    return Y

- We can construct the input tensor `X` in :numref:`fig_pooling` to **validate the output of the two-dimensional max-pooling layer**.


In [3]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
pool2d(X, (2, 2))

tensor([[4., 5.],
        [7., 8.]])

- Also, we can experiment with **the average pooling layer**.


In [4]:
pool2d(X, (2, 2), 'avg')

tensor([[2., 3.],
        [5., 6.]])

## [**Padding and Stride**]

- Just like convolutional layers, **pooling layers modify the output shape**.

- We can **control the output size** by:
  - **Padding** the input, and
  - **Adjusting the stride** of the pooling window.

- This is useful when we want to:
  - Maintain a particular **spatial resolution**, or
  - **Downsample** more aggressively.

- To demonstrate this, we use the **built-in 2D max-pooling layer** from a deep learning framework.

- Example setup:
  - Construct an **input tensor `X`** with **four dimensions**:
    - **Batch size = 1**
    - **Number of channels = 1**
    - Followed by **height and width**


In [5]:
X = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))
X

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]]]])

- Since **pooling aggregates** information over regions,
  - **Deep learning frameworks** often **default to using the same size** for:
    - The **pooling window**, and
    - The **stride**.

- For example:
  - If we use a pooling window of shape `(3, 3)`,
  - Then the default **stride** will also be `(3, 3)`.

- This setup ensures that:
  - The pooling regions **do not overlap**,
  - And the **entire input is covered** efficiently.


In [6]:
pool2d = nn.MaxPool2d(3)
# Pooling has no model parameters, hence it needs no initialization
pool2d(X)

tensor([[[[10.]]]])

- **The stride and padding can be manually specified** to override framework defaults if required.


In [7]:
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]]]])

- We can specify an arbitrary rectangular pooling window with **arbitrary height and width** respectively, as the example below shows.


In [8]:
pool2d = nn.MaxPool2d((2, 3), stride=(2, 3), padding=(0, 1))
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]]]])

## Multiple Channels

- When working with **multi-channel input**, the **pooling layer** handles each channel **independently**.

- That is, the pooling operation is applied to **each channel separately**, rather than combining them as in a convolutional layer.

- As a result:
  - The **number of output channels** is **equal to the number of input channels**.

- Example:
  - We can construct a two-channel input by concatenating:
    - Tensor `X`, and
    - Tensor `X + 1`,
  - Along the **channel dimension**.


In [9]:
X = torch.cat((X, X + 1), 1)
X

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]],

         [[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])

- As we can see, the number of output channels is still two after pooling.


In [10]:
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]],

         [[ 6.,  8.],
          [14., 16.]]]])

## Summary

- **Pooling** is a simple operation that **aggregates values** over a local window.

- All familiar **convolution semantics** still apply:
  - **Stride**
  - **Padding**

- **Channel-wise behavior**:
  - Pooling is applied to **each channel independently**.
  - It **does not change** the number of channels.

- Among the common pooling strategies:
  - **Max-pooling** is generally **preferred** over average pooling.
  - It provides a degree of **translation invariance**.

- A popular setting:
  - A **$2 \times 2$ max-pooling** window,
  - Reduces spatial resolution by **a factor of 4**.

- **Beyond basic pooling**:
  - Advanced alternatives exist:
    - **Stochastic pooling** :cite:`Zeiler.Fergus.2013`
    - **Fractional max-pooling** :cite:`Graham.2014`
  - These add **randomization** and may slightly improve performance.

- In **attention mechanisms**, aggregation can be more **adaptive**:
  - For example, based on **query–key alignment scores**.

